# Notebook 04: Final Evaluation & Report Generation

**Purpose:** Aggregate all results, build the comparison tables (Base vs SFT-winner vs DPO-winner), and produce report-ready artifacts.

**Inputs:** All files in `results/`

**Outputs:**
- `results/final_comparison.json` - structured side-by-side data
- `results/comparison_table.csv` - for pasting into the report
- `results/qualitative_examples.md` - side-by-side response examples per prompt


## Step 1: Setup and load everything

In [ ]:
import sys
import json
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path("/kaggle/working/daa-helper") if KAGGLE else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from utils.io_helpers import load_json, save_json

baseline = load_json("baseline_base_model.json", base_dir="results")
sft_winner_info = load_json("sft_winner.json", base_dir="results")
dpo_winner_info = load_json("dpo_winner.json", base_dir="results")

sft_winner_full = load_json(f"sft_{sft_winner_info['winning_trial']}.json", base_dir="results")
dpo_winner_full = load_json(f"dpo_{dpo_winner_info['winning_trial']}.json", base_dir="results")

# Load all trials for the full table
sft_config = load_json("sft_trials.json", base_dir="configs")
dpo_config = load_json("dpo_trials.json", base_dir="configs")

all_sft_results = []
for trial in sft_config["trials"]:
    try:
        all_sft_results.append(load_json(f"sft_{trial['name']}.json", base_dir="results"))
    except FileNotFoundError:
        pass

all_dpo_results = []
for trial in dpo_config["trials"]:
    try:
        all_dpo_results.append(load_json(f"dpo_{trial['name']}.json", base_dir="results"))
    except FileNotFoundError:
        pass

print(f"Loaded baseline + {len(all_sft_results)} SFT trials + {len(all_dpo_results)} DPO trials")


## Step 2: Build the master comparison table

In [ ]:
import csv

rows = []

# Base
rows.append({
    "Stage": "Base",
    "Trial": "qwen2.5-1.5b-instruct (4-bit)",
    "LoRA_r": "-",
    "Targets": "-",
    "LR": "-",
    "Batch": "-",
    "Epochs": "-",
    "Beta": "-",
    "Mean_BLEU": baseline["evaluation"]["aggregate"]["mean_bleu"],
    "Mean_BERTScore_F1": baseline["evaluation"]["aggregate"]["mean_bertscore_f1"],
    "Combined_Score": baseline["evaluation"]["aggregate"]["combined_score"],
    "Eval_Loss": "-",
    "Train_Time_s": "-",
})

for r in all_sft_results:
    c = r["config"]
    rows.append({
        "Stage": "SFT",
        "Trial": r["trial_name"],
        "LoRA_r": c["lora_r"],
        "Targets": str(c["target_modules"])[:30],
        "LR": c["learning_rate"],
        "Batch": c["per_device_train_batch_size"] * c["gradient_accumulation_steps"],
        "Epochs": c["num_train_epochs"],
        "Beta": "-",
        "Mean_BLEU": r["evaluation"]["aggregate"]["mean_bleu"],
        "Mean_BERTScore_F1": r["evaluation"]["aggregate"]["mean_bertscore_f1"],
        "Combined_Score": r["evaluation"]["aggregate"]["combined_score"],
        "Eval_Loss": r["training_metrics"].get("eval_loss"),
        "Train_Time_s": r["training_metrics"].get("train_runtime_seconds"),
    })

for r in all_dpo_results:
    c = r["config"]
    rows.append({
        "Stage": "DPO",
        "Trial": r["trial_name"],
        "LoRA_r": "(from SFT winner)",
        "Targets": "(from SFT winner)",
        "LR": c["learning_rate"],
        "Batch": c["per_device_train_batch_size"] * c["gradient_accumulation_steps"],
        "Epochs": c["num_train_epochs"],
        "Beta": c["beta"],
        "Mean_BLEU": r["evaluation"]["aggregate"]["mean_bleu"],
        "Mean_BERTScore_F1": r["evaluation"]["aggregate"]["mean_bertscore_f1"],
        "Combined_Score": r["evaluation"]["aggregate"]["combined_score"],
        "Eval_Loss": r["training_metrics"].get("eval_loss"),
        "Train_Time_s": r["training_metrics"].get("train_runtime_seconds"),
    })

# Save CSV
csv_path = PROJECT_ROOT / "results" / "comparison_table.csv"
with open(csv_path, "w", newline="") as f:
    if rows:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

print(f"Saved comparison table to {csv_path}")
print(f"\n{'Stage':<6} {'Trial':<30} {'BLEU':>7} {'BERTScore':>10} {'Combined':>10}")
for row in rows:
    bleu = row["Mean_BLEU"]
    bs = row["Mean_BERTScore_F1"]
    cs = row["Combined_Score"]
    bleu_s = f"{bleu:.2f}" if isinstance(bleu, (int, float)) else str(bleu)
    bs_s = f"{bs:.4f}" if isinstance(bs, (int, float)) else str(bs)
    cs_s = f"{cs:.4f}" if isinstance(cs, (int, float)) else str(cs)
    print(f"{row['Stage']:<6} {str(row['Trial'])[:30]:<30} {bleu_s:>7} {bs_s:>10} {cs_s:>10}")


## Step 3: Build qualitative comparison markdown (Base vs SFT vs DPO per prompt)

In [ ]:
lines = ["# Qualitative Comparison: Base vs SFT-winner vs DPO-winner\n"]
lines.append(f"- **Base model**: Qwen2.5-1.5B-Instruct (4-bit quantized)")
lines.append(f"- **SFT winner**: {sft_winner_info['winning_trial']}")
lines.append(f"- **DPO winner**: {dpo_winner_info['winning_trial']}")
lines.append("")

base_responses = baseline["sample_responses"]
sft_responses = sft_winner_full["sample_responses"]
dpo_responses = dpo_winner_full["sample_responses"]

for i in range(len(base_responses)):
    prompt = base_responses[i]["prompt"]
    gold = base_responses[i]["gold"]
    base_r = base_responses[i]["response"]
    sft_r = sft_responses[i]["response"]
    dpo_r = dpo_responses[i]["response"]

    lines.append(f"\n## Prompt {i+1}\n")
    lines.append(f"**Question:** {prompt}\n")
    lines.append(f"\n**Gold (Claude.ai):**\n\n{gold}\n")
    lines.append(f"\n**Base model response:**\n\n{base_r}\n")
    lines.append(f"\n**SFT model response:**\n\n{sft_r}\n")
    lines.append(f"\n**DPO model response:**\n\n{dpo_r}\n")
    lines.append("\n---")

qual_path = PROJECT_ROOT / "results" / "qualitative_examples.md"
with open(qual_path, "w") as f:
    f.write("\n".join(lines))

print(f"Saved qualitative comparison to {qual_path}")
print(f"({len(lines)} lines, {len(base_responses)} prompts compared)")


## Step 4: Save consolidated final_comparison.json (one file for the report)

In [ ]:
final = {
    "base": {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "quantization": "4-bit nf4",
        "evaluation": baseline["evaluation"]["aggregate"]
    },
    "sft_all_trials": [
        {"name": r["trial_name"], "config": r["config"],
         "aggregate_eval": r["evaluation"]["aggregate"],
         "training_metrics": r["training_metrics"]}
        for r in all_sft_results
    ],
    "sft_winner": {
        "name": sft_winner_info["winning_trial"],
        "config": sft_winner_full["config"],
        "aggregate_eval": sft_winner_full["evaluation"]["aggregate"],
        "training_metrics": sft_winner_full["training_metrics"],
        "selection_reason": "Highest combined BLEU+BERTScore on 10 test prompts; tie-break on lower eval loss."
    },
    "dpo_all_trials": [
        {"name": r["trial_name"], "config": r["config"],
         "aggregate_eval": r["evaluation"]["aggregate"],
         "training_metrics": r["training_metrics"]}
        for r in all_dpo_results
    ],
    "dpo_winner": {
        "name": dpo_winner_info["winning_trial"],
        "config": dpo_winner_full["config"],
        "aggregate_eval": dpo_winner_full["evaluation"]["aggregate"],
        "training_metrics": dpo_winner_full["training_metrics"],
        "selection_reason": "Highest combined BLEU+BERTScore on 10 test prompts; tie-break on lower eval loss."
    },
    "progression": {
        "base_combined":         baseline["evaluation"]["aggregate"]["combined_score"],
        "sft_winner_combined":   sft_winner_full["evaluation"]["aggregate"]["combined_score"],
        "dpo_winner_combined":   dpo_winner_full["evaluation"]["aggregate"]["combined_score"],
        "improvement_sft_over_base":  sft_winner_full["evaluation"]["aggregate"]["combined_score"] - baseline["evaluation"]["aggregate"]["combined_score"],
        "improvement_dpo_over_sft":  dpo_winner_full["evaluation"]["aggregate"]["combined_score"] - sft_winner_full["evaluation"]["aggregate"]["combined_score"],
    }
}

save_json(final, "final_comparison.json")

print("=" * 70)
print("FINAL PROGRESSION (combined score)")
print("=" * 70)
p = final["progression"]
print(f"Base model:        {p['base_combined']:.4f}")
print(f"SFT winner:        {p['sft_winner_combined']:.4f}  (Δ {p['improvement_sft_over_base']:+.4f})")
print(f"DPO winner:        {p['dpo_winner_combined']:.4f}  (Δ {p['improvement_dpo_over_sft']:+.4f})")


## Step 5: Generate a starter report skeleton

In [ ]:
report_md = f'''# DAA Helper: Fine-tuning Qwen2.5-1.5B for Algorithmic Reasoning Tutoring

**Authors:** [Member1, Member2]
**Course:** NLP with Deep Learning - Assignment 04
**Track:** 1 (LLM Fine-Tuning) - Option A (SFT → DPO)

## Abstract

We fine-tuned Qwen2.5-1.5B-Instruct to behave as a Design and Analysis of Algorithms (DAA) tutor that guides students through algorithmic reasoning step-by-step rather than dumping final solutions. Using a 2-stage pipeline (SFT on CodeForces editorials, then DPO on synthetic Socratic preference pairs), the combined BLEU+BERTScore on our 10-prompt held-out evaluation improved from {final["progression"]["base_combined"]:.4f} (base) → {final["progression"]["sft_winner_combined"]:.4f} (best SFT) → {final["progression"]["dpo_winner_combined"]:.4f} (best DPO).

## 1. Platform Details

- **Hardware:** Kaggle Notebooks, 2× NVIDIA T4 GPUs (16 GB VRAM each)
- **OS / Frameworks:** Linux, PyTorch, transformers (≥4.45), TRL (≥0.11), PEFT, bitsandbytes (4-bit QLoRA)
- **Quota used:** [fill in actual hours]

## 2. Data Details

### 2.1 Manual Test Set (10 prompts)
Hand-written DAA prompts covering: Master Theorem, loop complexity, algorithm comparison, recursion trees, amortized analysis, space-time tradeoffs, bottleneck identification, asymptotic notation, greedy vs DP, recurrence derivation. Gold answers were generated via Claude.ai with explicit Socratic-style instructions.

### 2.2 SFT Dataset
- **Source:** `open-r1/codeforces-cots` (HuggingFace), subset `solutions_w_editorials`
- **Why:** Editorials are contest-organizer-written tutorial explanations - exactly the explanatory style we want to teach the model.
- **Subset used:** 3,000 train + 200 eval (random sampled with seed=42)
- **Preprocessing:** Used existing `messages` column (already in chat format)

### 2.3 DPO Preference Dataset
- **Source:** Custom-generated using Claude.ai
- **Method:** 150 DAA question stubs auto-generated from templated categories (complexity, recurrences, comparisons, data structures, algorithm design, proof correctness, lower bounds). For each, Claude.ai produced a Socratic-guidance response (chosen) and an answer-dump response (rejected).
- **Size:** ~150 pairs (90/10 train/eval split)
- **Justification for size:** Behavioral steering with DPO is well-established with 100-200 pairs on small models; volume is less important than signal quality.

## 3. Experimentation, Analysis, and Insight

### 3.1 Model and Tokenizer
- **Base:** `Qwen/Qwen2.5-1.5B-Instruct` (4-bit nf4 quantization, double-quant, bf16 compute)
- **Tokenizer:** Qwen2 tokenizer with chat template

### 3.2 Evaluation Metrics
- **Primary:** Sentence-level BLEU (sacrebleu) + BERTScore F1 (deberta-xlarge-mnli), averaged over the 10 test prompts. Combined score = 0.5·(BLEU/100) + 0.5·BERTScore_F1.
- **Tie-breaker:** Lower validation loss.

### 3.3 SFT Trial Matrix (5 trials)

[INSERT Table 1: comparison_table.csv first 6 rows - base + 5 SFT]

**Winner:** {sft_winner_info["winning_trial"]} ({sft_winner_full["config"]["description"]})
- Mean BLEU: {sft_winner_full["evaluation"]["aggregate"]["mean_bleu"]:.2f}
- Mean BERTScore F1: {sft_winner_full["evaluation"]["aggregate"]["mean_bertscore_f1"]:.4f}
- Eval loss: {sft_winner_full["training_metrics"]["eval_loss"]:.4f}

### 3.4 DPO Trial Matrix (5 trials)

[INSERT Table 2: comparison_table.csv last 5 rows - 5 DPO]

**Winner:** {dpo_winner_info["winning_trial"]} ({dpo_winner_full["config"]["description"]})
- Mean BLEU: {dpo_winner_full["evaluation"]["aggregate"]["mean_bleu"]:.2f}
- Mean BERTScore F1: {dpo_winner_full["evaluation"]["aggregate"]["mean_bertscore_f1"]:.4f}
- Beta: {dpo_winner_full["config"]["beta"]}

### 3.5 Base vs SFT vs DPO Behavior

[INSERT EXAMPLES FROM qualitative_examples.md - pick 3-4 illustrative prompts]

**Observation patterns to look for:**
- **Base model:** Tends to either dump a solution immediately or fail to reason through the problem correctly
- **SFT model:** Produces longer, more explanation-style answers in the style of CodeForces editorials but still often presents the full solution upfront
- **DPO model:** Adopts more guidance-style language; uses questions and breaks problems into steps; reveals less of the final answer

### 3.6 Best Hyperparameters

| Parameter | Value |
|---|---|
| LoRA rank (SFT) | {sft_winner_full["config"]["lora_r"]} |
| LoRA target modules | {sft_winner_full["config"]["target_modules"]} |
| SFT learning rate | {sft_winner_full["config"]["learning_rate"]} |
| SFT batch size (effective) | {sft_winner_full["config"]["per_device_train_batch_size"] * sft_winner_full["config"]["gradient_accumulation_steps"]} |
| SFT epochs | {sft_winner_full["config"]["num_train_epochs"]} |
| DPO beta | {dpo_winner_full["config"]["beta"]} |
| DPO learning rate | {dpo_winner_full["config"]["learning_rate"]} |
| DPO epochs | {dpo_winner_full["config"]["num_train_epochs"]} |

### 3.7 Resource Usage and Training Time
[FILL FROM training_metrics fields in results]

### 3.8 Strengths and Weaknesses of SFT vs DPO

**SFT** taught the model the *vocabulary and structure* of algorithmic explanations - it learned to produce editorial-style content. But it could not separate "explain how to solve" from "give the solution".

**DPO** specifically targeted the *behavioral preference* - to prefer the Socratic style over the answer-dump style. This is exactly what DPO is designed for (preference between two valid completions), and it's why we get an additional gain on top of SFT.

### 3.9 Failure Cases
[FILL IN AFTER RUNNING - look for: hallucinated complexity bounds, premature solution reveals, repetition, incoherent multi-step reasoning]

## 4. Reproducibility

- All code, data, and notebooks are in our GitHub repo: [LINK]
- Random seed: 42 (data shuffle and DPO question generation)
- Exact package versions are pinned in `requirements.txt`
- LoRA adapter checkpoints uploaded to HuggingFace Hub: [LINK]
- Pipeline runs in 4 sequential notebooks (00, 02, 03, 04) with notebook 01 for manual DPO pair generation

## 5. References

1. open-r1 team. CodeForces-CoTs dataset. HuggingFace, 2025.
2. Rafailov et al. Direct Preference Optimization. NeurIPS 2023.
3. Hu et al. LoRA: Low-Rank Adaptation. ICLR 2022.
4. Dettmers et al. QLoRA: Efficient Finetuning of Quantized LLMs. NeurIPS 2023.
5. Qwen Team. Qwen2.5 Technical Report. 2024.
6. von Werra et al. TRL: Transformer Reinforcement Learning library. HuggingFace.

## Appendix

[Add screenshots of training curves, additional examples, etc.]
'''

report_path = PROJECT_ROOT / "results" / "report_skeleton.md"
with open(report_path, "w") as f:
    f.write(report_md)

print(f"Saved report skeleton to {report_path}")
print("Open this file, fill in the [INSERT...] and [FILL IN...] sections from the other result files,")
print("then convert to Word/PDF for submission.")


## Done!

In [ ]:
print("=" * 70)
print("✓ Pipeline complete.")
print("=" * 70)
print("\nKey artifacts for the report:")
print("  - results/final_comparison.json        # all metrics in one structured file")
print("  - results/comparison_table.csv         # for Tables 1 & 2 in report")
print("  - results/qualitative_examples.md      # base vs SFT vs DPO outputs per prompt")
print("  - results/report_skeleton.md           # fill-in-the-blanks report draft")
print("\nNext steps:")
print("  1. Fill in the [INSERT...] sections in report_skeleton.md")
print("  2. Convert to docx (or write directly in Word)")
print("  3. Rename: <Member1>_<Member2>.docx")
print("  4. Submit to LMS (NOT Google Drive)")
print("  5. Push code to GitHub and put link in report")
